In [2]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

sys.path.insert(0, str(PROJECT_ROOT))


# ------------------------------------------------------------
# Load clinical notes
# ------------------------------------------------------------

notes = pd.read_csv(
    DATA_DIR / "raw" / "clinical_notes.csv"
)

print("Original notes:", len(notes))
print("Columns:", notes.columns.tolist())

Original notes: 1602
Columns: ['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']


In [4]:
# ------------------------------------------------------------
# Remove broken notes
# ------------------------------------------------------------

notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

print(
    "After #NAME? removal:",
    len(notes_clean)
)


# ------------------------------------------------------------
# Deduplicate repeated note content per patient
# ------------------------------------------------------------

notes_dedup = (
    notes_clean
    .sort_values(
        [
            "person_id",
            "creation_timestamp",
        ]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "After deduplication:",
    len(notes_dedup)
)

print(
    "Patients:",
    notes_dedup["person_id"].nunique()
)

After #NAME? removal: 1595
After deduplication: 1103
Patients: 50


In [5]:
import re


def split_note_into_sections(note_text):
    """
    Split a clinical note using explicit section headings already
    present in the note.

    Returns:
        list of dictionaries:
        [
            {
                "section_name": "...",
                "chunk_text": "..."
            },
            ...
        ]
    """

    text = str(note_text).strip()

    if not text:
        return []

    section_names = [
        "Chief Complaint",
        "Presenting Complaint",
        "History of Present Illness",
        "HPI",
        "Past Medical History",
        "PMH",
        "Past Surgical History",
        "PSH",
        "Medications",
        "Current Medications",
        "Allergies",
        "Family History",
        "Social History",
        "Review of Systems",
        "ROS",
        "Physical Examination",
        "Physical Exam",
        "Examination",
        "Vital Signs",
        "Vitals",
        "Investigations",
        "Laboratory Results",
        "Labs",
        "Imaging",
        "Assessment",
        "Impression",
        "Diagnosis",
        "Diagnoses",
        "Plan",
        "Assessment and Plan",
        "Treatment",
        "Hospital Course",
        "Clinical Course",
        "Discharge Plan",
        "Follow Up",
        "Follow-Up",
    ]

    heading_pattern = "|".join(
        re.escape(name)
        for name in sorted(
            section_names,
            key=len,
            reverse=True
        )
    )

    pattern = re.compile(
        rf"(?im)^[ \t]*(?P<heading>{heading_pattern})"
        rf"[ \t]*(?::|-)?[ \t]*$"
    )

    matches = list(pattern.finditer(text))

    # If no recognized headings exist, keep the full note.
    if not matches:
        return [
            {
                "section_name": "Unsectioned",
                "chunk_text": text,
            }
        ]

    sections = []

    # Preserve text before the first heading.
    prefix = text[:matches[0].start()].strip()

    if prefix:
        sections.append(
            {
                "section_name": "Preamble",
                "chunk_text": prefix,
            }
        )

    # Extract each section.
    for i, match in enumerate(matches):

        section_name = match.group("heading").strip()

        content_start = match.end()

        if i + 1 < len(matches):
            content_end = matches[i + 1].start()
        else:
            content_end = len(text)

        content = text[
            content_start:content_end
        ].strip()

        if content:
            sections.append(
                {
                    "section_name": section_name,
                    "chunk_text": (
                        f"{section_name}\n{content}"
                    ),
                }
            )

    return sections

In [6]:
from sentence_transformers import SentenceTransformer

bge_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

print("BGE model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE model loaded.


In [7]:
section_records = []

for _, row in notes_dedup.iterrows():
    sections = split_note_into_sections(
        row["clean_note_text"]
    )

    for section in sections:
        section_records.append({
            "person_id": row["person_id"],
            "creation_timestamp": row["creation_timestamp"],
            "section_name": section["section_name"],
            "chunk_text": section["chunk_text"],
        })


section_chunks = pd.DataFrame(
    section_records
)

section_chunks = (
    section_chunks
    .sort_values(
        [
            "person_id",
            "creation_timestamp",
        ]
    )
    .reset_index(drop=True)
)

section_chunks["chunk_id"] = range(
    len(section_chunks)
)

print(
    "Total section chunks:",
    len(section_chunks)
)

print(
    "Patients:",
    section_chunks["person_id"].nunique()
)

display(section_chunks.head(10))

Total section chunks: 2771
Patients: 50


,person_id,creation_timestamp,section_name,chunk_text,chunk_id
0,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 08:45,Unsectioned,"- Patient: Tomos Ellis, 15-year-old male, pres...",0
1,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 09:10,Unsectioned,"Patient: Tomos Ellis, 15-year-old male, presen...",1
2,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 09:25,Unsectioned,"Patient name: Tomos Ellis, 15-year-old male. N...",2
3,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:00,Unsectioned,"Reviewed abdominal X-ray on 2026-01-07, which ...",3
4,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Preamble,Patient\nTomos Ellis\n\nAge\n15\n\nSex\nMale\n...,4
5,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Presenting Complaint,Presenting Complaint\nAbdominal pain\n\nHistor...,5
6,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Past Medical History,Past Medical History\nNil,6
7,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Medications,Medications\nNil,7
8,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Allergies,Allergies\nNKDA,8
9,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Social History,Social History\nLives at home with parents. At...,9


In [8]:
section_texts = (
    section_chunks["chunk_text"]
    .astype(str)
    .tolist()
)

section_embeddings = bge_model.encode(
    section_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(
    "Section embeddings shape:",
    section_embeddings.shape
)

Batches:   0%|          | 0/87 [00:00<?, ?it/s]

Section embeddings shape: (2771, 768)


In [9]:
RAG_QUERY = """
Retrieve the clinically relevant information needed to produce a
comprehensive longitudinal summary of this patient's clinical history,
including major diagnoses, treatments, investigations, clinical
progression, and outcomes.
""".strip()

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

TEST_PATIENT_ID = "04df53ea-55c1-48d9-84a1-1f15c133b29b"

# ------------------------------------------------------------
# Get this patient's chunks
# ------------------------------------------------------------

patient_mask = (
    section_chunks["person_id"] == TEST_PATIENT_ID
)

patient_chunks = (
    section_chunks.loc[patient_mask]
    .copy()
)

patient_embeddings = section_embeddings[
    patient_mask.to_numpy()
]

print("Patient chunks:", len(patient_chunks))


# ------------------------------------------------------------
# Embed the frozen retrieval query
# ------------------------------------------------------------

query_embedding = bge_model.encode(
    [RAG_QUERY],
    normalize_embeddings=True
)

print(
    "Query embedding shape:",
    query_embedding.shape
)


# ------------------------------------------------------------
# Cosine similarity
# ------------------------------------------------------------

similarities = cosine_similarity(
    query_embedding,
    patient_embeddings
)[0]

patient_chunks["similarity"] = similarities


# ------------------------------------------------------------
# Retrieve Top-20
# ------------------------------------------------------------

top_k = min(
    20,
    len(patient_chunks)
)

retrieved_sections = (
    patient_chunks
    .sort_values(
        "similarity",
        ascending=False
    )
    .head(top_k)
    .copy()
)

retrieved_sections["retrieval_rank"] = range(
    1,
    len(retrieved_sections) + 1
)


# ------------------------------------------------------------
# Reorder chronologically for summarization
# ------------------------------------------------------------

retrieved_sections = (
    retrieved_sections
    .sort_values(
        [
            "creation_timestamp",
            "chunk_id",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Retrieved sections:",
    len(retrieved_sections)
)

display(
    retrieved_sections[
        [
            "creation_timestamp",
            "section_name",
            "retrieval_rank",
            "similarity",
            "chunk_text",
        ]
    ]
)

Patient chunks: 71
Query embedding shape: (1, 768)
Retrieved sections: 20


,creation_timestamp,section_name,retrieval_rank,similarity,chunk_text
0,01/01/2026 07:30,Unsectioned,16,0.519921,"67M, Allan Victor Robinson (DOB: 1956-03-15, N..."
1,01/01/2026 08:00,Unsectioned,14,0.523391,67M. GCS 12/15 - disoriented to time/place. BP...
2,01/01/2026 08:30,Unsectioned,3,0.571885,"- Patient: Allan Victor R obinson, 67-year-old..."
3,01/01/2026 10:15,Unsectioned,20,0.516246,"10:15, 01/01/26 - Patient Allan Victor Robinso..."
4,01/01/2026 10:45,Past Medical History,2,0.604555,Past Medical History\n- HTN\n- OA
5,01/01/2026 11:30,Past Medical History,1,0.604555,Past Medical History\n- HTN\n- OA
6,01/01/2026 15:30,Unsectioned,12,0.529359,Physiotherapy assessment conducted on 01/01/26...
7,01/01/2026 16:00,Unsectioned,13,0.525739,NOK Maargaret Robinson phoned at 16:00. Update...
8,01/01/2026 17:00,Unsectioned,7,0.534261,Occupational therapy assessment conducted on 0...
9,02/01/2026 14:30,Unsectioned,17,0.519002,- Asesssment conducted by Therapist Haruto Wat...


In [11]:
rag_context_parts = []

for _, row in retrieved_sections.iterrows():

    rag_context_parts.append(
        f"[{row['creation_timestamp']}] "
        f"[{row['section_name']}]\n"
        f"{row['chunk_text']}"
    )

rag_context = "\n\n".join(
    rag_context_parts
)

print(
    "Retrieved chunks:",
    len(retrieved_sections)
)

print(
    "Context characters:",
    len(rag_context)
)

print(
    "Context words:",
    len(rag_context.split())
)

print("\n--- CONTEXT PREVIEW ---\n")
print(rag_context[:3000])

Retrieved chunks: 20
Context characters: 10078
Context words: 1446

--- CONTEXT PREVIEW ---

[01/01/2026 07:30] [Unsectioned]
67M, Allan Victor Robinson (DOB: 1956-03-15, NHS: 680034032), arrived A&E 07:30 w/ acute confusion post minor fall. Triage by Nurse Shaun Patrick Hopkins. Vitals: BP 170/95, HR 88. No allergies. PMHx: HTN, OA. No meds. Assigned Cat 2 due to confusion & recent trauma. ED dx: Subdural hygroma. Preped for further assess & invx. Awaiting revi ew.
Nurse Shaun Patrick Hopkins 
NMC number: 05J2149V

[01/01/2026 08:00] [Unsectioned]
67M. GCS 12/15 - disoriented to time/place. BP 170/95, pulse 88 bpm. Triaged by Nurse Shaun Patrick Hopkins. Chief complaint: acute confusion after minor fall. Triage categody: 2. PMH: HTN, Osteoarthritis. No known drug allergies. Referred for urgent bloods  + CT head to r/o ICI.
Nurse Shaun Patrick Hopkins 
NMC number: 05J2149V

[01/01/2026 08:30] [Unsectioned]
- Patient: Allan Victor R obinson, 67-year-old male, NHS number 680034032. 
 - D

In [12]:
from config.prompts import SUMMARY_PROMPT
from src.llm.llm import generate_rag_summary

In [14]:
import src.llm.llm as llm_module

print("Generation model:", llm_module.LLM_MODEL)

Generation model: gpt-5.6-luna


In [15]:
section_bge_summary = generate_rag_summary(
    context=rag_context,
    prompt=SUMMARY_PROMPT
)

print(section_bge_summary)

67-year-old man with past medical history of hypertension and osteoarthritis, on no regular medications and with no known drug allergies, presented to A&E on 01/01/2026 after a minor fall with acute confusion. Initial observations were BP 170/95 and HR 88; GCS was 12/15 with disorientation to time and place. Urgent blood tests (FNC, electrolytes and coagulation profile) were sent, and CT head demonstrated a subdural hygroma. He remained haemodynamically and respiratorily stable, with no external injuries, and was referred for admission to the neurology ward.

On 01/01, physiotherapy noted mild confusion and slight gait instability; daily balance and mobility therapy was planned. Occupational therapy identified mild impairment in fine motor coordination affecting buttons and utensils, with therapy planned for ADLs and home safety. The next of kin was updated.

On 02/01, speech and language assessment found an adequate swallow without aspiration and appropriate speech comprehension/artic

In [16]:
# ============================================================
# LOAD THE SAME 50-PATIENT STUDY COHORT
# ============================================================

STUDY_PATIENT_IDS_PATH = (
    PROJECT_ROOT
    / "data"
    / "patients"
    / "study_patient_ids.json"
)

with STUDY_PATIENT_IDS_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    STUDY_PATIENT_IDS = json.load(file)

print("Study patients:", len(STUDY_PATIENT_IDS))

assert len(STUDY_PATIENT_IDS) == 50

Study patients: 50


In [17]:
# ============================================================
# RAG OUTPUT CHECKPOINT
# ============================================================

RAG_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "rag"
    / "final_rag_summaries.json"
)

RAG_RESULTS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

if RAG_RESULTS_PATH.exists():
    with RAG_RESULTS_PATH.open(
        "r",
        encoding="utf-8"
    ) as file:
        rag_results = json.load(file)
else:
    rag_results = {}

print(
    "Already completed:",
    len(rag_results)
)

Already completed: 0


In [18]:
# ============================================================
# RUN RAG FOR ALL 50 PATIENTS
# ============================================================

total_patients = len(STUDY_PATIENT_IDS)

for i, patient_id in enumerate(
    STUDY_PATIENT_IDS,
    start=1
):

    # Skip if already saved
    if patient_id in rag_results:
        print(
            f"[{i}/{total_patients}] "
            f"Skipping completed: {patient_id}"
        )
        continue

    print(
        f"[{i}/{total_patients}] "
        f"Processing: {patient_id}"
    )

    # --------------------------------------------------------
    # 1. Get patient-specific chunks
    # --------------------------------------------------------

    patient_mask = (
        section_chunks["person_id"] == patient_id
    )

    patient_chunks = (
        section_chunks.loc[patient_mask]
        .copy()
    )

    patient_embeddings = section_embeddings[
        patient_mask.to_numpy()
    ]


    # --------------------------------------------------------
    # 2. Embed frozen RAG query
    # --------------------------------------------------------

    query_embedding = bge_model.encode(
        [RAG_QUERY],
        normalize_embeddings=True
    )


    # --------------------------------------------------------
    # 3. Cosine similarity
    # --------------------------------------------------------

    similarities = cosine_similarity(
        query_embedding,
        patient_embeddings
    )[0]

    patient_chunks["similarity"] = similarities


    # --------------------------------------------------------
    # 4. Retrieve Top-20
    # --------------------------------------------------------

    top_k = min(
        20,
        len(patient_chunks)
    )

    retrieved_sections = (
        patient_chunks
        .sort_values(
            "similarity",
            ascending=False
        )
        .head(top_k)
        .copy()
    )

    retrieved_sections["retrieval_rank"] = range(
        1,
        len(retrieved_sections) + 1
    )


    # --------------------------------------------------------
    # 5. Reorder chronologically
    # --------------------------------------------------------

    retrieved_sections = (
        retrieved_sections
        .sort_values(
            [
                "creation_timestamp",
                "chunk_id",
            ]
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 6. Build RAG context
    # --------------------------------------------------------

    rag_context_parts = []

    for _, row in retrieved_sections.iterrows():

        rag_context_parts.append(
            f"[{row['creation_timestamp']}] "
            f"[{row['section_name']}]\n"
            f"{row['chunk_text']}"
        )

    rag_context = "\n\n".join(
        rag_context_parts
    )


    # --------------------------------------------------------
    # 7. Generate final summary
    # --------------------------------------------------------

    start = time.time()

    summary = generate_rag_summary(
        context=rag_context,
        prompt=SUMMARY_PROMPT
    )

    latency = time.time() - start


    # --------------------------------------------------------
    # 8. Save this patient
    # --------------------------------------------------------

    rag_results[patient_id] = {
        "person_id": patient_id,
        "source_chunk_count": len(
            patient_chunks
        ),
        "retrieved_chunk_count": len(
            retrieved_sections
        ),
        "context_characters": len(
            rag_context
        ),
        "context_words": len(
            rag_context.split()
        ),
        "summary": summary,
        "latency_seconds": round(
            latency,
            2
        ),
    }


    # --------------------------------------------------------
    # 9. Checkpoint immediately
    # --------------------------------------------------------

    with RAG_RESULTS_PATH.open(
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            rag_results,
            file,
            indent=2,
            ensure_ascii=False
        )


    print(
        f"Saved | "
        f"retrieved={len(retrieved_sections)} | "
        f"latency={latency:.2f}s"
    )

    time.sleep(2)

[1/50] Processing: 028998ee-babc-4096-9b28-001bc2f9a84e
Saved | retrieved=20 | latency=6.05s
[2/50] Processing: 04df53ea-55c1-48d9-84a1-1f15c133b29b
Saved | retrieved=20 | latency=9.25s
[3/50] Processing: 05192757-942f-460d-b4ff-004ec39cc5ee
Saved | retrieved=20 | latency=9.42s
[4/50] Processing: 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf
Saved | retrieved=20 | latency=8.07s
[5/50] Processing: 0f438665-d430-4adb-8acc-c3beed9e4942
Saved | retrieved=20 | latency=8.27s
[6/50] Processing: 136c7916-4f9b-4e5c-bf01-77e9d2c681a2
Saved | retrieved=20 | latency=9.19s
[7/50] Processing: 137b8481-4f1d-4b7f-babd-20f7117023ad
Saved | retrieved=20 | latency=6.91s
[8/50] Processing: 1705dd0f-011a-492c-b006-b27e03f2f4ed
Saved | retrieved=20 | latency=10.27s
[9/50] Processing: 1dbe23dc-0d1e-431b-81eb-497282b46a14
Saved | retrieved=20 | latency=10.82s
[10/50] Processing: 28570119-9cdc-4120-98c0-4edb76cf36a3
Saved | retrieved=20 | latency=7.04s
[11/50] Processing: 29ea304f-821d-474e-81a1-394ca3945e02
Saved | re

In [19]:
print(
    "Completed RAG summaries:",
    len(rag_results)
)

assert len(rag_results) == 50

Completed RAG summaries: 50
